In [ ]:
from abc import ABC, abstractmethod
from decimal import Decimal


from mainnet_launch.constants import ChainData, ETH_CHAIN
from multicall import Call
from mainnet_launch.data_fetching.get_state_by_block import (
    get_state_by_one_block,
    build_blocks_to_use,
    get_raw_state_by_blocks,
)
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"

SECONDS_PER_YEAR = 365 * 24 * 60 * 60


def get_seconds_between_blocks(
    chain: ChainData,
    start_block: int,
    end_block: int,
) -> int:
    start_timestamp = chain.client.eth.get_block(start_block)["timestamp"]
    end_timestamp = chain.client.eth.get_block(end_block)["timestamp"]
    return end_timestamp - start_timestamp


class IncentiveTokenPricer(ABC):

    def __init__(self, chain: ChainData, denominated_in: str):
        self.chain = chain
        self.denominated_in = denominated_in

    @abstractmethod
    def get_incentive_token_prices(self, incentive_token_addresses: list[str], block: int) -> dict[str, float]:
        pass


class ConstantIncentiveTokenPricer(IncentiveTokenPricer):
    def __init__(self, chain, denominated_in, prices: dict[str, float]):
        super().__init__(chain, denominated_in)
        self.prices = prices

    def get_incentive_token_prices(self, incentive_token_addresses: list[str], block: int) -> float:
        return {token: self.prices.get(token, 0) for token in incentive_token_addresses}


class LpTokenPricer(ABC):
    def __init__(self, chain: ChainData, denominated_in: str):
        self.chain = chain
        self.denominated_in = denominated_in

    @abstractmethod
    def get_lp_token_price(self, lp_token_address: str, block: int) -> float:
        pass


class ConstantLPTokenPricer(LpTokenPricer):
    def __init__(self, chain: ChainData, denominated_in: str, constant_value: float):
        super().__init__(chain, denominated_in)
        self.constant_value = constant_value

    def get_lp_token_price(self, lp_token_address: str, block: int) -> float:
        return self.constant_value


class BacktestIncentiveAprCalculator(ABC):

    def __init__(
        self,
        chain: ChainData,
        lp_token_address: str,
        incentive_token_pricer: IncentiveTokenPricer,
        lp_token_pricer: LpTokenPricer,
    ):
        self.chain = chain
        self.lp_token_address = lp_token_address
        self.incentive_token_pricer = incentive_token_pricer
        self.lp_token_pricer = lp_token_pricer

    def _compute_incentive_token_delta(self, start_block: int, end_block: int) -> dict[str, int]:
        """
        Computes the change in claimable incentive tokens per 1 LP token between two blocks.

        eg at the start of block we can get 0.01 CRV per LP token and at the end of block we can get 0.015 CRV per LP token, then the delta is +0.005 CRV per LP token.
        """
        start_claimable = self._compute_one_lp_token_claimable_amount(start_block)
        end_claimable = self._compute_one_lp_token_claimable_amount(end_block)
        delta = {
            token: end_claimable.get(token, 0) - start_claimable.get(token, 0)
            for token in set(start_claimable) | set(end_claimable)
        }
        return delta

    @abstractmethod
    def _compute_one_lp_token_claimable_amount(self, block: int) -> dict[str, int]:
        """
        Returns the amount of incentive token claimable per 1 LP token at a given block.
        """
        pass

    def compute_incentive_apr(
        self,
        start_block: int,
        end_block: int,
    ) -> float:
        """
        Returns the incentive apr earnd by a destination in portion form. eg .01 for 1% APR.
        """
        assumed_token_prices = self.incentive_token_pricer.get_incentive_token_prices(
            ["wstETH", "BAL", "AURA"], end_block
        )
        assumed_lp_token_value = self.lp_token_pricer.get_lp_token_price(self.lp_token_address, end_block)

        incentive_token_delta = self._compute_incentive_token_delta(start_block, end_block)
        extra_incentive_token_value: float = sum(
            amount * assumed_token_prices.get(token, 0) for token, amount in incentive_token_delta.items()
        )

        seconds_between_blocks = get_seconds_between_blocks(self.chain, start_block, end_block)
        portion_of_year = seconds_between_blocks / SECONDS_PER_YEAR
        incentive_apr = (extra_incentive_token_value / assumed_lp_token_value) / portion_of_year

        return incentive_apr


class AuraIncentiveAprBacktestCalculator(BacktestIncentiveAprCalculator):
    def __init__(
        self,
        chain: ChainData,
        lp_token_address: str,
        incentive_token_pricer: IncentiveTokenPricer,
        lp_token_pricer: LpTokenPricer,
        calls: list[Call],
    ):
        super().__init__(chain, lp_token_address, incentive_token_pricer, lp_token_pricer)
        self.calls = calls

    def _compute_one_lp_token_claimable_amount(self, block: int) -> dict[str, int]:
        claimable_amounts = get_state_by_one_block(self.calls, block, self.chain)

        for k, v in claimable_amounts.items():
            if v is None:
                claimable_amounts[k] = 0
        return claimable_amounts


# https://app.aura.finance/#/1/pool/254
# toy prices from
#  https://www.coingecko.com/en/coins/wrapped-steth/eth
# and
# https://ca.finance.yahoo.com/quote/BAL-ETH/?guccounter=1
# using the oct 1 2025 prices


def norm_1e18(success, x):
    if success:
        return x / 1e18


def identify_function(success, x):
    if success:
        return x


pool_address = "0x6b31a94029fd7840d780191B6D63Fa0D269bd883"
reward_pool_address = "0x5CDa24fE2d80703DBaaB7217eF1Ef8101161e271"  # BaseRewardPool4626
extra_rewards_0 = "0x17C016eCCA8c9b865f7f77ec412f0c660C9FB896"  # rewwardPoolAddress.extraRewards(0)
extra_rewards_1 = "0x6a0647724431cE3499baB6B95509451f62333154"  # rewardPoolAddress.extraRewards(1)


reward_per_token_calls = [
    Call(reward_pool_address, ["rewardPerToken()(uint256)"], [["BAL", norm_1e18]]),  # .rewardToken() -> BAL
    Call(extra_rewards_0, ["rewardPerToken()(uint256)"], [["AURA", norm_1e18]]),  # .rewardToken() -> stash token Aura
    Call(
        extra_rewards_1, ["rewardPerToken()(uint256)"], [["wstETH", norm_1e18]]
    ),  # .rewardToken() -> stash token  wstETH
]


assumed_token_prices = {"BAL": 0.000263, "AURA": 0.00004237067, "wstETH": 1.12154}
assumed_lp_token_value = 1.021

aura_incentive_apr_calculator = AuraIncentiveAprBacktestCalculator(
    chain=ETH_CHAIN,
    lp_token_address=pool_address,
    incentive_token_pricer=ConstantIncentiveTokenPricer(ETH_CHAIN, "ETH", assumed_token_prices),
    lp_token_pricer=ConstantLPTokenPricer(ETH_CHAIN, "ETH", assumed_lp_token_value),
    calls=reward_per_token_calls,
)

# a block at the end of each day since the pool was deployed

In [32]:
blocks = build_blocks_to_use(start_block=22124378, chain=ETH_CHAIN)[::7]

records = []
for i, start_block in enumerate(blocks[:-1]):
    end_block = blocks[i + 1]
    incentive_apr = aura_incentive_apr_calculator.compute_incentive_apr(start_block, end_block)
    start_datetime = pd.to_datetime(ETH_CHAIN.client.eth.get_block(start_block)["timestamp"], unit="s", utc=True)
    records.append(
        {
            "start_datetime": start_datetime,
            "start_block": start_block,
            "end_block": end_block,
            "incentive_apr": incentive_apr,
        }
    )

weekly_incentive_apr_df = pd.DataFrame.from_records(records).sort_values("end_block")
px.line(
    weekly_incentive_apr_df,
    x="start_datetime",
    y="incentive_apr",
    title="Weekly Surge Fluid wstETH/WETH Balancer Pool Incentive APR",
)

In [ ]:
# Extract September 1, 2025 and October 1, 2025 data
sep_1_2025 = df[df.index.date == pd.Timestamp("2025-09-01").date()]
oct_1_2025 = df[df.index.date == pd.Timestamp("2025-10-01").date()]

sep_1_dict = sep_1_2025.iloc[0].to_dict()
oct_1_dict = oct_1_2025.iloc[0].to_dict()

print("September 1, 2025:")
print(sep_1_dict)
print("\nOctober 1, 2025:")
print(oct_1_dict)
claimable_df = get_raw_state_by_blocks(reward_per_token_calls, blocks, ETH_CHAIN, include_block_number=True)
px.line(
    claimable_df[["BAL", "AURA", "wstETH"]],
    title="Reward Per Token Over Time Aura Surge wstETH:ETH fluid",
)

AttributeError: 'RangeIndex' object has no attribute 'date'